In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
datasetPath = Path("/home/philipm/Desktop/ML-For-CV-Robustness/datasets/azimuth-datasets/comp-gt-azim.csv")

#making sure
datasetPath.is_file()

True

In [3]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,azimuth,grviOut
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,-0.605931,0.2
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,-0.605931,0.2
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,-0.605931,0.2
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,-0.605931,0.2
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,-0.605931,0.2


In [4]:
target_col = dataset["grviOut"]
labels = target_col.to_numpy(dtype="float32")
labels

array([0.2 , 0.2 , 0.2 , ..., 0.26, 0.26, 0.26],
      shape=(667240,), dtype=float32)

In [5]:
inputs = dataset.drop("grviOut", axis=1)
inputs = inputs.to_numpy(dtype="float32")
inputs

array([[ 1.63549468e-01,  3.24192282e-04,  4.20509488e-04, ...,
         1.02023224e+03,  0.00000000e+00, -6.05931282e-01],
       [ 9.71098840e-02,  4.11663583e-04,  4.86339268e-04, ...,
         1.02023224e+03,  1.00000000e+00, -6.05931282e-01],
       [ 7.43446946e-02,  4.47855098e-04,  5.17319597e-04, ...,
         1.02023224e+03,  2.00000000e+00, -6.05931282e-01],
       ...,
       [ 2.83671737e-01,  2.69046490e-04,  4.22214536e-04, ...,
         4.83019989e+02,  1.70000000e+01,  1.30001771e+00],
       [ 2.41321996e-01,  3.44786560e-04,  5.07728080e-04, ...,
         4.83019989e+02,  1.80000000e+01,  1.30001771e+00],
       [ 2.74991482e-01,  3.26303503e-04,  5.12171478e-04, ...,
         4.83019989e+02,  1.90000000e+01,  1.30001771e+00]],
      shape=(667240, 7), dtype=float32)

In [6]:
if len(labels) != len (inputs):
    print ("problem")

In [7]:
n_total = len(inputs)

n_train = int(0.8*n_total)
n_test = int(0.1*n_total)
n_val = int(0.1*n_total)

# shuffle
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n_total, generator=g).numpy()

inputs_shuf = inputs[perm]
labels_shuf = labels[perm]

# split
train_inputs, train_labels = inputs_shuf[:n_train], labels_shuf[:n_train]
val_inputs, val_labels = inputs_shuf[n_train:n_train + n_val], labels_shuf[n_train:n_train + n_val]
test_inputs, test_labels = inputs_shuf[n_train + n_val:], labels_shuf[n_train + n_val:]

In [8]:
# scaling
from sklearn.preprocessing import QuantileTransformer, StandardScaler

if True:
    input_scaler = StandardScaler() #QuantileTransformer(output_distribution='normal')
    target_scaler = StandardScaler()

    train_inputs = input_scaler.fit_transform(train_inputs)
    train_labels = target_scaler.fit_transform(train_labels.reshape(-1, 1)).reshape(-1)

    val_inputs = input_scaler.transform(val_inputs)
    val_labels = target_scaler.transform(val_labels.reshape(-1, 1)).reshape(-1)

    test_inputs = input_scaler.transform(test_inputs)
    test_labels = target_scaler.transform(test_labels.reshape(-1, 1)).reshape(-1)

In [9]:
train_inputs = torch.from_numpy(train_inputs).float()
train_labels = torch.from_numpy(train_labels).float()

val_inputs = torch.from_numpy(val_inputs).float()
val_labels = torch.from_numpy(val_labels).float()

test_inputs = torch.from_numpy(test_inputs).float()
test_labels = torch.from_numpy(test_labels).float()

In [10]:
print ("Train size:", len(train_inputs))
print ("Val size:", len(val_inputs))
print ("Test size:", len(test_inputs))

Train size: 533792
Val size: 66724
Test size: 66724


In [11]:
BATCH_SIZE = 512

class metricsDataset (Dataset):
    def __init__(self, metrics, labels):
        self.metrics = metrics
        self.labels = labels

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], self.labels[idx]

train_dataset = metricsDataset(train_inputs, train_labels)
val_dataset   = metricsDataset(val_inputs, val_labels)
test_dataset  = metricsDataset(test_inputs, test_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    pin_memory=True
)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [13]:
model = nn.Sequential(
    nn.Linear(7, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 1)
).to(device)

In [14]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    T_mult=1,
    eta_min=1e-6
)

In [15]:
EPOCHS = 160

best_val_loss = float("inf")

for epoch in range(0, EPOCHS):
    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        optimizer.zero_grad()

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        preds = model(x).squeeze(-1)
        loss = criterion(preds, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            pred = model(x).squeeze(-1)
            loss = criterion(pred, y)

            val_loss += loss.item() * x.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoch {epoch + 1:3d}/{EPOCHS} | "
        f"Train MSE: {train_loss:.6f} | "
        f"Val MSE: {val_loss:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        # SAVE the model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, 'best_model_checkpoint.pth')

        print("Best Model Saved")

checkpoint = torch.load('best_model_checkpoint.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model (Val Loss: {checkpoint['best_val_loss']})")

model.eval()
test_loss = 0.0
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        pred = model(x).squeeze(-1)
        loss = criterion(pred, y)
        test_loss += loss.item() * x.size(0)

test_loss /= len(test_loader.dataset)
print(f"\nTest MSE: {test_loss:.6f}")
print(f"Test RMSE: {test_loss ** 0.5:.6f}")

Epoch   1/160 | Train MSE: 0.812657 | Val MSE: 0.858617
Best Model Saved
Epoch   2/160 | Train MSE: 0.701229 | Val MSE: 0.768720
Best Model Saved
Epoch   3/160 | Train MSE: 0.675346 | Val MSE: 1.041503
Epoch   4/160 | Train MSE: 0.662236 | Val MSE: 0.846950
Epoch   5/160 | Train MSE: 0.654486 | Val MSE: 0.912507
Epoch   6/160 | Train MSE: 0.649045 | Val MSE: 0.827190
Epoch   7/160 | Train MSE: 0.646211 | Val MSE: 0.903156
Epoch   8/160 | Train MSE: 0.642506 | Val MSE: 0.794491
Epoch   9/160 | Train MSE: 0.640153 | Val MSE: 0.866787
Epoch  10/160 | Train MSE: 0.634800 | Val MSE: 0.692213
Best Model Saved
Epoch  11/160 | Train MSE: 0.627754 | Val MSE: 0.746316
Epoch  12/160 | Train MSE: 0.624300 | Val MSE: 0.724769
Epoch  13/160 | Train MSE: 0.620402 | Val MSE: 0.747119
Epoch  14/160 | Train MSE: 0.613661 | Val MSE: 0.654238
Best Model Saved
Epoch  15/160 | Train MSE: 0.614083 | Val MSE: 0.783480
Epoch  16/160 | Train MSE: 0.609311 | Val MSE: 0.688775
Epoch  17/160 | Train MSE: 0.599848 

In [16]:
rel_test_loss = (test_loss**0.5)*100 / (test_labels.max()-test_labels.min())
print(str(float(rel_test_loss)) + "%")

14.419733047485352%
